In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read cleaned GDP data from Silver Lakehouse
gdp_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.worldbank_gdp_clean"
)

print("Silver GDP records:", gdp_df.count())

# Define previous-year window
year_window = Window.partitionBy("country_code").orderBy("year")

# Build Gold GDP dimension
dim_gdp = (
    gdp_df
    .withColumn(
        "previous_year_gdp_usd",
        F.lag("gdp_current_usd").over(year_window)
    )
    .withColumn(
        "gdp_growth_percent",
        F.when(
            F.col("previous_year_gdp_usd").isNotNull()
            & (F.col("previous_year_gdp_usd") != 0),
            F.round(
                (
                    (F.col("gdp_current_usd") - F.col("previous_year_gdp_usd"))
                    / F.col("previous_year_gdp_usd")
                ) * 100,
                2
            )
        )
    )
    .withColumn("gdp_key", F.col("year").cast("int"))
    .select(
        "gdp_key",
        "year",
        "country_code",
        "country_name",
        "indicator",
        "gdp_current_usd",
        "previous_year_gdp_usd",
        "gdp_growth_percent"
    )
    .orderBy("year")
)

# Save to Gold Lakehouse
(
    dim_gdp.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimGDP")
)

print("Gold GDP dimension saved successfully!")
print("Gold records:", dim_gdp.count())

display(dim_gdp.orderBy(F.desc("year")))

StatementMeta(, b72d12fc-87e8-4700-be81-6b9cf09e9f57, 3, Finished, Available, Finished, False)

Silver GDP records: 66
Gold GDP dimension saved successfully!
Gold records: 66


SynapseWidget(Synapse.DataFrame, 02a3cd29-45a8-4495-ab5d-1d6c56894a3d)